# 4. BM25 + Phi-3 query expansion

Local-versus-hosted query-expansion comparison; no API needed for local inference.

This is a recorded full-train comparison, not a new evaluation.

In [1]:
from pathlib import Path
import json,sys
import pandas as pd
ROOT=next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'src/retrieval.py').exists())
sys.path.insert(0,str(ROOT/'src'))
sys.path.insert(0,str(ROOT/'scripts'))
from presentation_review import selection,cached_demo
model=selection(4)
model['config_snapshot']

{'name': 'llm_local_phi3_rewrite',
 'provider': 'ollama',
 'model': 'phi3:mini',
 'model_revision': '4f222292793889a9a40a020799cfd28d53f3e01af25d48e06c5e708610fc47e9',
 'runtime': 'ollama-0.40.13',
 'prompt_file': 'prompts/retrieval_rewrite_v1.txt',
 'temperature': 0,
 'seed': 42,
 'max_output_tokens': 256,
 'context_length': 2048,
 'baseline_config': 'configs/bm25_focused_stemmed.json',
 'expose_in_demo': True}

In [2]:
m=model['train_metrics']
table=pd.DataFrame(m['retrieval'])[['k','n_queries','precision','recall','hit']]
table

,k,n_queries,precision,recall,hit
0,1,140,0.180864,0.454660,0.550000
1,3,140,0.112357,0.637666,0.735714
2,5,140,0.080682,0.707500,0.800000
3,10,140,0.052631,0.814167,0.871429


In [3]:
print('Correct negative abstention:',m['negative_counts'])
print('False answerable abstention:',m['false_abstention'])
print('Cached latency:',m['latency_ms'])

Correct negative abstention: {'n': 23, 'correct': 0}
False answerable abstention: 0.0
Cached latency: {'median': 11.98835001559928, 'p95': 15.204055042704562}


## Cached example

The question comes from recorded train predictions. Config settings and original source offsets remain visible; the model cannot call a hosted provider here.

In [4]:
demo=cached_demo(model)
print(demo['query'])
print(demo.get('summary') or demo['message'])
pd.DataFrame(demo['hits'])[['file_path','start','end','text']]

Consider the Amendment No. 3 to Wireless Content License Agreement between Fox Mobile Entertainment, Inc. and Glu Mobile Inc.; What is the expiration date of this contract?
2 passages retrieved.


,file_path,start,end,text
0,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,0,1000,"AMENDMENT NO. 3\n\nDated as of February 19, 20..."
1,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,1600,1902,"x Mobile Entertainment, Inc. (""Fox"")\n\n\n\nBy..."


Full code walkthrough: ../experiments/11_local_phi3_comparison.ipynb. Historical acceptance and current preference are separate; see REPORT.md for limitations.